In [ ]:
!pip install transformers==4.45.2 accelerate
!pip install git+https://github.com/maxidl/marker-arena.git@ffeb6ee6c1092f1e008000cb8d1d6240a7baeb52

  Cloning https://github.com/maxidl/marker-arena.git (to revision ffeb6ee6c1092f1e008000cb8d1d6240a7baeb52) to /tmp/pip-req-build-zs1_ixmt
  Running command git clone --filter=blob:none --quiet https://github.com/maxidl/marker-arena.git /tmp/pip-req-build-zs1_ixmt
  Running command git rev-parse -q --verify 'sha^ffeb6ee6c1092f1e008000cb8d1d6240a7baeb52'
  Running command git fetch -q https://github.com/maxidl/marker-arena.git ffeb6ee6c1092f1e008000cb8d1d6240a7baeb52
  Resolved https://github.com/maxidl/marker-arena.git to commit ffeb6ee6c1092f1e008000cb8d1d6240a7baeb52
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer
import torch
from threading import Thread

from marker.convert import convert_single_pdf
from marker.output import markdown_exists, save_markdown, get_markdown_filepath
from marker.pdf.utils import find_filetype
from marker.pdf.extract_text import get_length_of_text
from marker.models import load_all_models
from marker.settings import settings
from marker.logger import configure_logging
from surya.settings import settings as surya_settings
import traceback

In [ ]:

# Define prompts
SYSTEM_PROMPT_TEMPLATE = """You are an expert reviewer for AI conferences. You follow best practices and review papers according to the reviewer guidelines.
Reviewer guidelines:
1. Read the paper: It’s important to carefully read through the entire paper, and to look up any related work and citations that will help you comprehensively evaluate it. Be sure to give yourself sufficient time for this step.
2. While reading, consider the following:
    - Objective of the work: What is the goal of the paper? Is it to better address a known application or problem, draw attention to a new application or problem, or to introduce and/or explain a new theoretical finding? A combination of these? Different objectives will require different considerations as to potential value and impact.
    - Strong points: is the submission clear, technically correct, experimentally rigorous, reproducible, does it present novel findings (e.g. theoretically, algorithmically, etc.)?
    - Weak points: is it weak in any of the aspects listed in b.?
    - Be mindful of potential biases and try to be open-minded about the value and interest a paper can hold for the community, even if it may not be very interesting for you.
3. Answer four key questions for yourself, to make a recommendation to Accept or Reject:
    - What is the specific question and/or problem tackled by the paper?
    - Is the approach well motivated, including being well-placed in the literature?
    - Does the paper support the claims? This includes determining if results, whether theoretical or empirical, are correct and if they are scientifically rigorous.
    - What is the significance of the work? Does it contribute new knowledge and sufficient value to the community? Note, this does not necessarily require state-of-the-art results. Submissions bring value to the community when they convincingly demonstrate new, relevant, impactful knowledge (incl., empirical, theoretical, for practitioners, etc).
4. Write your review including the following information:
    - Summarize what the paper claims to contribute. Be positive and constructive.
    - List strong and weak points of the paper. Be as comprehensive as possible.
    - Clearly state your initial recommendation (accept or reject) with one or two key reasons for this choice.
    - Provide supporting arguments for your recommendation.
    - Ask questions you would like answered by the authors to help you clarify your understanding of the paper and provide the additional evidence you need to be confident in your assessment.
    - Provide additional feedback with the aim to improve the paper. Make it clear that these points are here to help, and not necessarily part of your decision assessment.
Your write reviews in markdown format. Your reviews contain the following sections:
# Review
{review_fields}
Your response must only contain the review in markdown format with sections as defined above.
"""

USER_PROMPT_TEMPLATE = """Review the following paper:
{paper_text}
"""

# For now, use fixed review fields
REVIEW_FIELDS = """## Summary
Briefly summarize the paper and its contributions. This is not the place to critique the paper; the authors should generally agree with a well-written summary.
## Soundness
Please assign the paper a numerical rating on the following scale to indicate the soundness of the technical claims, experimental and research methodology and on whether the central claims of the paper are adequately supported with evidence. Choose from the following:
4: excellent
3: good
2: fair
1: poor
## Presentation
Please assign the paper a numerical rating on the following scale to indicate the quality of the presentation. This should take into account the writing style and clarity, as well as contextualization relative to prior work. Choose from the following:
4: excellent
3: good
2: fair
1: poor
## Contribution
Please assign the paper a numerical rating on the following scale to indicate the quality of the overall contribution this paper makes to the research area being studied. Are the questions being asked important? Does the paper bring a significant originality of ideas and/or execution? Are the results valuable to share with the broader ICLR community? Choose from the following:
4: excellent
3: good
2: fair
1: poor
## Strengths
A substantive assessment of the strengths of the paper, touching on each of the following dimensions: originality, quality, clarity, and significance. We encourage reviewers to be broad in their definitions of originality and significance. For example, originality may arise from a new definition or problem formulation, creative combinations of existing ideas, application to a new domain, or removing limitations from prior results.
## Weaknesses
A substantive assessment of the weaknesses of the paper. Focus on constructive and actionable insights on how the work could improve towards its stated goals. Be specific, avoid generic remarks. For example, if you believe the contribution lacks novelty, provide references and an explanation as evidence; if you believe experiments are insufficient, explain why and exactly what is missing, etc.
## Questions
Please list up and carefully describe any questions and suggestions for the authors. Think of the things where a response from the author can change your opinion, clarify a confusion or address a limitation. This is important for a productive rebuttal and discussion phase with the authors.
## Flag For Ethics Review
If there are ethical issues with this paper, please flag the paper for an ethics review and select area of expertise that would be most useful for the ethics reviewer to have. Please select all that apply. Choose from the following:
No ethics review needed.
Yes, Discrimination / bias / fairness concerns
Yes, Privacy, security and safety
Yes, Legal compliance (e.g., GDPR, copyright, terms of use)
Yes, Potentially harmful insights, methodologies and applications
Yes, Responsible research practice (e.g., human subjects, data release)
Yes, Research integrity issues (e.g., plagiarism, dual submission)
Yes, Unprofessional behaviors (e.g., unprofessional exchange between authors and reviewers)
Yes, Other reasons (please specify below)
## Details Of Ethics Concerns
Please provide details of your concerns.
## Rating
Please provide an "overall score" for this submission. Choose from the following:
1: strong reject
3: reject, not good enough
5: marginally below the acceptance threshold
6: marginally above the acceptance threshold
8: accept, good paper
10: strong accept, should be highlighted at the conference
"""


In [ ]:

# functions
def create_messages(review_fields, paper_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_TEMPLATE.format(review_fields=review_fields)},
        {"role": "user", "content": USER_PROMPT_TEMPLATE.format(paper_text=paper_text)},
    ]
    return messages


In [ ]:

model_refs = load_all_models()
metadata = {}
MAX_PAGES = 12
MIN_LENGTH=200
def convert_file(filepath):
    full_text, images, out_metadata = convert_single_pdf(
            filepath, model_refs, metadata=metadata, max_pages=MAX_PAGES
    )
    return full_text

def process_file(filepath):
    paper_text = convert_file(filepath)
    paper_text = paper_text.strip()
    if not len(paper_text) > MIN_LENGTH:
        raise ValueError()
    return paper_text


Loaded detection model vikp/surya_det3 on device cuda with dtype torch.float16
Loaded detection model vikp/surya_layout3 on device cuda with dtype torch.float16
Loaded reading order model vikp/surya_order on device cuda with dtype torch.float16
Loaded recognition model vikp/surya_rec on device cuda with dtype torch.float16
Loaded texify model to cuda with torch.float16 dtype


/usr/local/lib/python3.12/dist-packages/transformers/models/auto/image_processing_auto.py:517: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(


In [ ]:
pipe = pipeline("text-generation", model="maxidl/Llama-OpenReviewer-8B", device="cuda", torch_dtype=torch.bfloat16)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
from transformers import pipeline

def generate(paper_text, review_template):
    messages = create_messages(review_template, paper_text)
    return pipe(messages, max_new_tokens=4096)

In [ ]:
import os
import tqdm
pdfs = os.listdir("pdfs")
os.makedirs("papers", exist_ok=True)

for pdf in tqdm.tqdm(pdfs):
  content = process_file(f"pdfs/{pdf}")
  content = content.replace("Published as a conference paper at ICLR", "")
  with open(f"papers/{pdf.replace('.pdf','.md')}", "w") as f:
    f.write(content)

In [ ]:
os.makedirs("reviews", exist_ok=True)
for pdf in tqdm.tqdm(pdfs):
  with open(f"papers/{pdf.replace('.pdf','.md')}", "r") as f:
    content = f.read()
  review = generate(content, REVIEW_FIELDS)
  with open(f"reviews/{pdf.replace('.pdf','.md')}", "w") as f:
    f.write(review[-1]["generated_text"][-1]["content"])

100%|██████████| 200/200 [49:57<00:00, 14.99s/it]


In [ ]:
!zip reviews.zip reviews -r

updating: reviews/ (stored 0%)
  adding: reviews/WhO6Km5Rku.md (deflated 57%)
  adding: reviews/GRufFX1gAy.md (deflated 57%)
  adding: reviews/Y9TgNFsNyP.md (deflated 55%)
  adding: reviews/GMlZt4fZSY.md (deflated 50%)
  adding: reviews/IU4rqTlpRb.md (deflated 52%)
  adding: reviews/Kw2mvnzCoc.md (deflated 52%)
  adding: reviews/32mrjmaeMP.md (deflated 63%)
  adding: reviews/4Ha2srdhPN.md (deflated 53%)
  adding: reviews/abgW71sKIt.md (deflated 50%)
  adding: reviews/HHYd4Pz5Lp.md (deflated 49%)
  adding: reviews/Me0n0iESJY.md (deflated 57%)
  adding: reviews/opU91paIvZ.md (deflated 64%)
  adding: reviews/NFB4QGGS65.md (deflated 54%)
  adding: reviews/sJxBWDc8SM.md (deflated 56%)
  adding: reviews/Vit5M0G5Gb.md (deflated 74%)
  adding: reviews/BGEdvJ35PV.md (deflated 60%)
  adding: reviews/rBj2iVyrhh.md (deflated 54%)
  adding: reviews/4VW9HVCRw0.md (deflated 63%)
  adding: reviews/cEXEmyW77N.md (deflated 55%)
  adding: reviews/Rj5ZJk956j.md (deflated 62%)
  adding: reviews/0cbUKCyBsH.